# 0. Descrição da metodologia

## Etapa 1 — disponibilidade e correlação entre estações pluviométricas

**Dados diários → totais mensais completos → disponibilidade → Pearson pareado → R² → meses coincidentes → ranking de apoio.**

`evaluation_stations` seleciona somente os alvos do ranking. As matrizes,
disponibilidade e gráficos mantêm todas as estações carregadas por `entrada.estacoes`,
que continuam candidatas a apoio. `n_support_stations: null` mantém todos os apoios
elegíveis; um inteiro limita a seleção aos primeiros N, após os critérios existentes.

Cada execução cria uma pasta `basin_<ID>_<DDMM>_<HHMMSS>` dentro de `saida.diretorio`,
com a hora atual no fuso configurado. O ID vem do caminho de entrada, e não é
necessário digitá-lo quando esse caminho identifica a bacia.

Este notebook contém toda a lógica. Edite `config.yaml`, mantenha os dois arquivos
na mesma pasta e execute as células em ordem. O CSV real deve estar acessível no
computador que executa o kernel. O caminho Windows do YAML é um exemplo fornecido
na especificação, não um arquivo incluído nesta entrega.

Instalação: `python -m pip install -r requirements.txt`. Inicie o Jupyter na pasta
dos arquivos: `python -m jupyterlab`. Alternativamente, defina a variável de ambiente
`CONFIG_PLUVIOMETRIA` com o caminho de outro YAML antes de iniciar o Jupyter.

### Regras e escolhas explícitas

- Cada total mensal exige **todos os dias do mês civil**, inclusive 29/02 em anos
  bissextos. Meses parcialmente abrangidos pelo período permanecem `NaN`; dias
  fora do período não são usados para completá-los.
- Zero é válido. Um dia ausente no arquivo é inserido no calendário **com `NaN`**;
  isso explicita uma lacuna e não preenche precipitação.
- Datas inválidas, repetidas, com fuso ou com horário diferente de meia-noite
  interrompem a execução. Não há deduplicação, deslocamento de datas nem soma de
  registros diários repetidos. A validação estrutural verifica o arquivo inteiro.
- Números inválidos, infinitos ou fora dos limites configurados interrompem por
  padrão. A opção explícita `marcar_nan` registra cada ocorrência e marca apenas
  a cópia de trabalho. Nenhum dado bruto é alterado.
- Cada par usa sua própria interseção de meses observados. Com menos de dois
  pares, ou série constante na interseção, `r` e `R²` são indefinidos (`NaN`).
- **A diagonal de `r` e `R²` é 1 por convenção de apresentação**, conforme a
  especificação, inclusive para estações vazias/constantes. Ela não comprova
  disponibilidade ou correlação calculável, e nunca participa da seleção.
- `R² = r²` descreve associação linear bivariada (equivalente ao R² de uma
  regressão simples com intercepto no mesmo conjunto de pares). Não é uma medida
  de desempenho de preenchimento, que não é realizado nesta etapa.
- O limiar de meses para **selecionar apoio** é independente do mínimo matemático
  de dois pares para exibir `r`. Poucos meses podem produzir `r` alto, mas não
  qualificam a estação se ela não cumprir ambos os limiares do YAML.
- A disponibilidade usa todos os meses civis tocados pelo período como
  denominador. Início/fim diários e início/fim mensais completos são informados
  separadamente; extremos temporais não significam uma série contínua.

A correlação dos totais mensais pode refletir sazonalidade comum e não garante
homogeneidade, proximidade física ou disponibilidade no mês de uma futura falha.
Este resultado é uma triagem para etapas posteriores. **Não há preenchimento de
falhas, regressão de preenchimento ou curva de dupla massa.**


## 1. Imports

Além de pandas, NumPy, PyYAML e Plotly, são usados módulos da biblioteca padrão
para caminhos, auditoria e identificação do ambiente. SciPy não é necessário:
Pearson é calculado pelo pandas, sem ajustar uma regressão adicional.


In [15]:
from pathlib import Path, PureWindowsPath
from datetime import datetime, timezone
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError
from copy import deepcopy
from importlib.metadata import version
from html import escape
import csv
import hashlib
import json
import logging
import os
import platform
import re

import numpy as np
import pandas as pd
import yaml
import plotly.graph_objects as go
from IPython.display import display


## 2. Leitura do YAML

Todos os caminhos relativos são resolvidos a partir da pasta do YAML, não do CSV.
O único nome de descoberta no código é `config.yaml`; a variável de ambiente
permite escolher outra configuração sem editar o notebook. Caminhos de unidades
Windows funcionam no Windows. Em Linux/Colab, a unidade precisa ser montada e o
caminho correspondente informado no YAML.

`saida.diretorio` é a pasta-base. `iniciar_auditoria` cria nela uma subpasta exclusiva
para esta execução, com ID extraído de `basin_<ID>_...` nas pastas de entrada
(ou, como alternativa, do nome do CSV). `saida.id_bacia` é uma opção manual apenas
para caminhos fora desse padrão. O timestamp da pasta de entrada não é reutilizado.
Usa-se `saida.fuso_horario`; se já existir o mesmo nome, acrescenta-se `_02`, `_03`,
etc., sem sobrescrever a execução anterior. A estrutura interna atual é mantida.

Os arquivos de auditoria são atualizados durante a execução. `status: concluido`
indica exportação bem-sucedida. Uma tentativa interrompida permanece identificada
na sua própria pasta.


In [16]:
def resolver_caminho(valor, base):
    """Resolve caminho do YAML; recebe texto/base e retorna Path absoluto."""
    if not isinstance(valor, str) or not valor.strip():
        raise ValueError("Caminho vazio ou não textual no YAML.")
    if os.name != "nt" and (PureWindowsPath(valor).drive or "\\" in valor):
        raise ValueError(
            f"Caminho Windows em kernel não Windows: {valor}. "
            "Monte a unidade e informe seu caminho local no YAML."
        )
    caminho = Path(valor).expanduser()
    return (caminho if caminho.is_absolute() else base / caminho).resolve()


def validar_selecao_alvos(selecao):
    """Valida all/lista de códigos textuais; recebe seleção e retorna None ou lança erro."""
    if isinstance(selecao, str) and selecao == "all":
        return
    if (not isinstance(selecao, list) or not selecao
            or any(not isinstance(s, str) or not s.strip() for s in selecao)):
        raise ValueError('evaluation_stations deve ser all ou uma lista não vazia de códigos entre aspas.')
    if len(set(selecao)) != len(selecao):
        raise ValueError('evaluation_stations contém códigos repetidos.')


def resolver_estacoes_avaliacao(selecao, disponiveis):
    """Resolve alvos sem filtrar apoios; recebe seleção/nomes disponíveis e retorna lista de strings."""
    validar_selecao_alvos(selecao)
    disponiveis = [str(s) for s in disponiveis]
    if isinstance(selecao, str):
        return disponiveis
    faltam = [s for s in selecao if s not in disponiveis]
    if faltam:
        raise ValueError(
            f'Estação(ões)-alvo não disponível(is): {faltam}. '
            'Confira os cabeçalhos do CSV e o filtro entrada.estacoes.'
        )
    return list(selecao)


def identificar_bacia(caminho, id_manual=None):
    """Extrai ID textual das pastas/arquivo; recebe caminho e alternativa manual, retorna string."""
    if id_manual is not None:
        if not isinstance(id_manual, str) or not re.fullmatch(r'\d+', id_manual):
            raise ValueError('saida.id_bacia deve ser null ou um código numérico entre aspas.')
        return id_manual
    # A análise textual também reconhece caminhos Windows em outros sistemas.
    partes = re.split(r'[\\/]+', str(caminho))
    ids = {m.group(1) for p in partes[:-1]
           if (m := re.match(r'^basin_(\d+)(?:_|$)', p))}
    if len(ids) > 1:
        raise ValueError(f'IDs de bacia ambíguos nas pastas: {sorted(ids)}. Confira o caminho ou saida.id_bacia.')
    if ids:
        return ids.pop()
    nome = re.search(r'(?:^|_)basin_(\d+)(?:_|\.|$)', partes[-1])
    if nome:
        return nome.group(1)
    raise ValueError(
        'Não foi possível identificar a bacia no caminho. Use uma pasta basin_<ID>_... '
        'ou informe saida.id_bacia no YAML.'
    )


def criar_diretorio_execucao(base, id_bacia, fuso_horario):
    """Cria pasta exclusiva com hora atual; recebe base/ID/fuso e retorna (Path, instante)."""
    instante = datetime.now(ZoneInfo(fuso_horario))
    base = Path(base)
    base.mkdir(parents=True, exist_ok=True)
    nome = f'basin_{id_bacia}_{instante:%d%m_%H%M%S}'
    tentativa = 1
    while True:
        pasta = base / (nome if tentativa == 1 else f'{nome}_{tentativa:02d}')
        try:
            # A criação exclusiva evita misturar execuções até no mesmo segundo.
            pasta.mkdir(exist_ok=False)
            return pasta, instante
        except FileExistsError:
            tentativa += 1


def carregar_configuracao(caminho):
    """Lê e valida o YAML; recebe seu caminho e retorna configurações resolvidas."""
    caminho = Path(caminho).expanduser().resolve()
    if not caminho.is_file():
        raise FileNotFoundError(f"YAML inexistente: {caminho}")
    cfg = yaml.safe_load(caminho.read_text(encoding="utf-8-sig"))
    esquema = {
        "projeto": ["nome"],
        "entrada": ["arquivo_csv", "coluna_data", "separador", "decimal", "encoding",
                    "formato_data", "estacoes", "valores_ausentes"],
        "periodo": ["inicio", "fim"],
        "validacao": ["acao_valores_invalidos", "precipitacao_minima_mm",
                      "precipitacao_maxima_mm", "meses_completos_alerta"],
        "selecao_apoio": ["correlacao_minima", "meses_coincidentes_minimos"],
        "visualizacao": ["exibir_no_notebook", "template", "altura_series", "altura_heatmap",
                         "mostrar_valores_ate_n_estacoes", "casas_decimais",
                         "escala_correlacao", "escala_r2", "escala_meses", "scroll_zoom"],
        "saida": ["diretorio", "sobrescrever", "separador_csv", "decimal_csv",
                  "encoding_csv", "representacao_nan"],
        "arquivos": ["mensal", "disponibilidade", "correlacao", "r2", "meses_coincidentes",
                     "estacoes_apoio", "series_html", "correlacao_html", "r2_html",
                     "meses_html", "diagnostico_pares", "auditoria", "log", "configuracao"],
    }
    if not isinstance(cfg, dict):
        raise ValueError("O YAML precisa conter um mapeamento de configurações.")
    for secao, campos in esquema.items():
        if not isinstance(cfg.get(secao), dict):
            raise ValueError(f"Seção ausente/inválida: {secao}")
        faltam = set(campos) - set(cfg[secao])
        if faltam:
            raise ValueError(f"Faltam parâmetros em {secao}: {sorted(faltam)}")

    # Ausência dos novos parâmetros mantém todas as estações e todos os apoios elegíveis.
    cfg.setdefault("evaluation_stations", "all")
    cfg.setdefault("n_support_stations", None)
    validar_selecao_alvos(cfg["evaluation_stations"])
    limite_apoios = cfg["n_support_stations"]
    if limite_apoios is not None and (type(limite_apoios) is not int or limite_apoios < 1):
        raise ValueError("n_support_stations deve ser null ou um inteiro >= 1.")
    cfg["saida"].setdefault("id_bacia", None)
    cfg["saida"].setdefault("fuso_horario", "America/Sao_Paulo")
    try:
        ZoneInfo(cfg["saida"]["fuso_horario"])
    except (ZoneInfoNotFoundError, TypeError, ValueError) as exc:
        raise ValueError("Fuso horário inválido ou indisponível. Confira saida.fuso_horario e instale tzdata.") from exc

    for secao, chave in [("entrada", "separador"), ("entrada", "decimal"),
                         ("saida", "separador_csv"), ("saida", "decimal_csv")]:
        if not isinstance(cfg[secao][chave], str) or len(cfg[secao][chave]) != 1:
            raise ValueError(f"{secao}.{chave} deve ser um único caractere.")
    for sep, dec in [(cfg["entrada"]["separador"], cfg["entrada"]["decimal"]),
                     (cfg["saida"]["separador_csv"], cfg["saida"]["decimal_csv"])]:
        if sep == dec:
            raise ValueError("Separador de campo e separador decimal devem ser diferentes.")
    for secao, chave in [("saida", "sobrescrever"), ("visualizacao", "exibir_no_notebook"),
                         ("visualizacao", "scroll_zoom")]:
        if not isinstance(cfg[secao][chave], bool):
            raise ValueError(f"{secao}.{chave} deve ser true ou false.")
    inteiros = [("selecao_apoio", "meses_coincidentes_minimos", 2),
                ("validacao", "meses_completos_alerta", 0),
                ("visualizacao", "mostrar_valores_ate_n_estacoes", 0),
                ("visualizacao", "casas_decimais", 0),
                ("visualizacao", "altura_series", 1),
                ("visualizacao", "altura_heatmap", 1)]
    for secao, chave, minimo in inteiros:
        valor = cfg[secao][chave]
        if type(valor) is not int or valor < minimo:
            raise ValueError(f"{secao}.{chave} deve ser inteiro >= {minimo}.")
    rmin = cfg["selecao_apoio"]["correlacao_minima"]
    if isinstance(rmin, bool) or not isinstance(rmin, (int, float)) or not -1 <= rmin <= 1:
        raise ValueError("correlacao_minima deve ser um número entre -1 e 1.")
    val = cfg["validacao"]
    if val["acao_valores_invalidos"] not in {"interromper", "marcar_nan"}:
        raise ValueError("acao_valores_invalidos: use interromper ou marcar_nan.")
    for chave in ["precipitacao_minima_mm", "precipitacao_maxima_mm"]:
        v = val[chave]
        if v is not None and (isinstance(v, bool) or not isinstance(v, (int, float)) or not np.isfinite(v)):
            raise ValueError(f"{chave} deve ser número finito ou null.")
    if val["precipitacao_minima_mm"] is None or val["precipitacao_minima_mm"] > 0:
        raise ValueError("precipitacao_minima_mm deve ser <= 0, preservando zero válido.")
    if val["precipitacao_maxima_mm"] is not None and val["precipitacao_maxima_mm"] < 0:
        raise ValueError("precipitacao_maxima_mm deve ser >= 0 ou null.")
    entrada = cfg["entrada"]
    if not isinstance(entrada["formato_data"], str) or not entrada["formato_data"]:
        raise ValueError("Informe formato_data explicitamente para evitar datas ambíguas.")
    if not isinstance(entrada["valores_ausentes"], list):
        raise ValueError("valores_ausentes deve ser uma lista.")
    escolhidas = entrada["estacoes"]
    if escolhidas is not None and (not isinstance(escolhidas, list) or not escolhidas
            or any(not isinstance(x, str) for x in escolhidas) or len(set(escolhidas)) != len(escolhidas)):
        raise ValueError("estacoes deve ser null ou lista não vazia de nomes textuais distintos.")

    limites = []
    for chave in ["inicio", "fim"]:
        dt = pd.to_datetime(str(cfg["periodo"][chave]), format="%Y-%m-%d", errors="raise")
        if pd.isna(dt) or dt.tzinfo is not None or dt != dt.normalize():
            raise ValueError(f"periodo.{chave} deve ser uma data sem horário/fuso.")
        limites.append(dt)
    if limites[0] > limites[1]:
        raise ValueError("O início do período deve ser anterior ou igual ao fim.")
    entrada_path = resolver_caminho(entrada["arquivo_csv"], caminho.parent)
    saida_path = resolver_caminho(cfg["saida"]["diretorio"], caminho.parent)
    destinos = {}
    for chave in esquema["arquivos"]:
        nome = cfg["arquivos"][chave]
        if not isinstance(nome, str) or not nome or Path(nome).name != nome or "/" in nome or "\\" in nome or ":" in nome:
            raise ValueError(f"arquivos.{chave} deve ser um nome simples de arquivo.")
        destino = (saida_path / nome).resolve()
        if destino in {entrada_path, caminho}:
            raise ValueError("Uma saída sobrescreveria o CSV bruto ou o YAML de entrada.")
        destinos[chave] = destino
    if len({str(p).casefold() for p in destinos.values()}) != len(destinos):
        raise ValueError("Os nomes de saída devem ser distintos.")
    cfg["_runtime"] = {"yaml": caminho, "entrada": entrada_path, "saida": saida_path,
                       "inicio": limites[0], "fim": limites[1], "destinos": destinos,
                       "saida_base": saida_path,
                       "id_bacia": identificar_bacia(entrada["arquivo_csv"], cfg["saida"]["id_bacia"])}
    return cfg


def iniciar_auditoria(cfg):
    """Cria saída e log; recebe configuração e retorna estado da auditoria."""
    rt = cfg["_runtime"]
    saida_anterior = rt["saida"]
    pasta, instante = criar_diretorio_execucao(rt["saida_base"], rt["id_bacia"], cfg["saida"]["fuso_horario"])
    rt["saida"] = pasta
    rt["instante_execucao"] = instante.isoformat()
    rt["destinos"] = {chave: pasta / destino.relative_to(saida_anterior)
                      for chave, destino in rt["destinos"].items()}
    existentes = [str(p) for p in rt["destinos"].values() if p.exists()]
    if existentes and not cfg["saida"]["sobrescrever"]:
        raise FileExistsError(f"Saídas já existem; altere o diretório ou sobrescrever no YAML: {existentes}")
    logger = logging.getLogger("etapa1_pluviometria")
    logger.setLevel(logging.INFO)
    logger.propagate = False
    for handler in list(logger.handlers):
        handler.close()
        logger.removeHandler(handler)
    for handler in [logging.StreamHandler(), logging.FileHandler(rt["destinos"]["log"], mode="w", encoding="utf-8")]:
        handler.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
        logger.addHandler(handler)
    estado = {"logger": logger, "arquivo": rt["destinos"]["auditoria"], "dados": {
        "status": "em_execucao", "inicio_utc": datetime.now(timezone.utc).isoformat(),
        "csv": str(rt["entrada"]), "yaml_origem": str(rt["yaml"]), "periodo": cfg["periodo"],
        "id_bacia": rt["id_bacia"], "instante_execucao": rt["instante_execucao"],
        "diretorio_execucao": str(rt["saida"]),
        "evaluation_stations": cfg["evaluation_stations"], "n_support_stations": cfg["n_support_stations"],
        "ambiente": {"python": platform.python_version(), **{p: version(p) for p in ["pandas", "numpy", "PyYAML", "plotly"]}},
        "eventos": []}}
    registrar(estado, "info", "inicio", "Auditoria iniciada; o CSV bruto não será modificado.",
              {"saidas_existentes": existentes})
    return estado


def registrar(estado, nivel, codigo, mensagem, detalhes=None):
    """Registra evento completo em JSON e mensagem no log; retorna None."""
    getattr(estado["logger"], nivel)(mensagem)
    estado["dados"]["eventos"].append({"codigo": codigo, "nivel": nivel,
        "mensagem": mensagem, "detalhes": detalhes})
    if nivel == "error":
        estado["dados"]["status"] = "falhou"
    estado["arquivo"].write_text(json.dumps(estado["dados"], ensure_ascii=False, indent=2, default=str), encoding="utf-8")


def falhar(estado, codigo, mensagem, detalhes=None):
    """Registra erro e detalhes na auditoria, depois lança ValueError."""
    registrar(estado, "error", codigo, mensagem, detalhes)
    raise ValueError(mensagem)


config = carregar_configuracao(os.environ.get("CONFIG_PLUVIOMETRIA", "config.yaml"))
auditoria = iniciar_auditoria(config)
print(f"Projeto: {config['projeto']['nome']}")
print(f"Entrada: {config['_runtime']['entrada']}")
print(f"Saída: {config['_runtime']['saida']}")


INFO | Auditoria iniciada; o CSV bruto não será modificado.


Projeto: Análise pluviométrica
Entrada: A:\Codigo\UNIPLU-BR\1.Evaluation_basin\resultados\basin_86500000_2409_154543\series\chuva_basin_86500000_test.csv
Saída: A:\Codigo\UNIPLU-BR\2. Analise_basin\resultados_consistencia\basin_86500000_2809_131052


## 3. Leitura e validação do CSV

Os cabeçalhos são preservados como texto (inclusive códigos com zeros à esquerda).
Colunas de metadados não devem entrar como precipitação: nesse caso, liste as
estações em `entrada.estacoes`. Esse filtro antigo restringe o universo carregado
(alvos **e** apoios). A nova chave `evaluation_stations`, na raiz do YAML, restringe
apenas os alvos. Use `all` ou uma lista de códigos entre aspas. Alvos inexistentes,
duplicados ou excluídos por `entrada.estacoes` causam erro claro. Cada registro deve representar um único dia.
Marcadores de ausência são definidos explicitamente no YAML; texto não numérico
não é convertido silenciosamente em precipitação ausente.


In [17]:
def validar_datas(serie, formato, estado):
    """Valida datas textuais de todo o CSV; retorna Series datetime sem correções."""
    datas = pd.to_datetime(serie, format=formato, errors="coerce")
    invalidas = datas.isna()
    if invalidas.any():
        detalhes = [{"registro_csv": int(i) + 2, "valor": str(serie.loc[i])} for i in serie.index[invalidas]]
        falhar(estado, "datas_invalidas", f"{len(detalhes)} data(s) inválida(s). Veja auditoria e formato_data.", detalhes)
    if not pd.api.types.is_datetime64_any_dtype(datas.dtype) or datas.dt.tz is not None:
        falhar(estado, "fuso_horario", "Datas com fuso ou fusos mistos não são aceitas; prepare a referência diária na fonte.")
    horario = datas.ne(datas.dt.normalize())
    duplicadas = datas.duplicated(keep=False)
    problemas = {}
    if horario.any():
        problemas["horarios_nao_nulos"] = [{"registro_csv": int(i) + 2, "data": str(datas.loc[i])} for i in datas.index[horario]]
    if duplicadas.any():
        problemas["datas_duplicadas"] = [{"registro_csv": int(i) + 2, "data": str(datas.loc[i])} for i in datas.index[duplicadas]]
    if problemas:
        falhar(estado, "estrutura_datas", "Datas duplicadas ou horários não diários encontrados; nenhuma linha foi removida. Veja auditoria.", problemas)
    registrar(estado, "info", "datas_validas", f"{len(datas)} datas válidas e sem duplicatas no CSV.")
    return datas


def carregar_dados(cfg, estado):
    """Lê e valida CSV/datas/valores; recebe configuração e auditoria, retorna DataFrame diário ordenado."""
    ent, caminho = cfg["entrada"], cfg["_runtime"]["entrada"]
    if not caminho.is_file():
        falhar(estado, "arquivo_inexistente", f"Arquivo CSV inexistente: {caminho}")
    try:
        with caminho.open("r", encoding=ent["encoding"], newline="") as arquivo:
            leitor = csv.reader(arquivo, delimiter=ent["separador"])
            cabecalho = next(leitor, None)
            if not cabecalho or any(not c.strip() for c in cabecalho):
                falhar(estado, "cabecalho_vazio", "Cabeçalho ausente ou com coluna sem nome.")
            if len(set(cabecalho)) != len(cabecalho):
                falhar(estado, "colunas_duplicadas", "Há nomes de colunas duplicados no cabeçalho.", cabecalho)
            estrutura = [{"registro_csv": i, "campos": len(row)} for i, row in enumerate(leitor, start=2)
                         if len(row) != len(cabecalho)]
            if estrutura:
                falhar(estado, "registros_malformados", "Registros CSV com número de campos diferente do cabeçalho.", estrutura)
        bruto = pd.read_csv(caminho, sep=ent["separador"], encoding=ent["encoding"],
                            dtype=str, keep_default_na=False, na_filter=False, skip_blank_lines=False)
    except (OSError, UnicodeError, pd.errors.ParserError, pd.errors.EmptyDataError, csv.Error) as exc:
        falhar(estado, "leitura_csv", f"Falha ao ler CSV: {exc}")
    if bruto.empty:
        falhar(estado, "csv_vazio", "O CSV não contém registros diários.")
    if ent["coluna_data"] not in bruto.columns:
        falhar(estado, "coluna_data_ausente", f"Coluna de data inexistente: {ent['coluna_data']}", list(bruto.columns))
    estacoes = ent["estacoes"] if ent["estacoes"] is not None else [c for c in bruto.columns if c != ent["coluna_data"]]
    if not estacoes or ent["coluna_data"] in estacoes or set(estacoes) - set(bruto.columns):
        falhar(estado, "estacoes_invalidas", "Seleção de estações vazia, inexistente ou incluindo a coluna de data.", estacoes)
    datas = validar_datas(bruto[ent["coluna_data"]], ent["formato_data"], estado)
    numericos = pd.DataFrame(index=bruto.index)
    invalidos, ausentes = [], {}
    marcadores = {str(x).strip() for x in ent["valores_ausentes"]} | {""}
    val = cfg["validacao"]
    for estacao in estacoes:
        texto = bruto[estacao].str.strip()
        falta = texto.isin(marcadores)
        normalizado = texto.mask(falta).str.replace(ent["decimal"], ".", regex=False)
        valores = pd.to_numeric(normalizado, errors="coerce").astype(float)
        invalido = (~falta) & (~np.isfinite(valores) | (valores < val["precipitacao_minima_mm"]))
        if val["precipitacao_maxima_mm"] is not None:
            invalido |= (~falta) & (valores > val["precipitacao_maxima_mm"])
        for i in bruto.index[invalido]:
            invalidos.append({"registro_csv": int(i) + 2, "data": str(datas.loc[i].date()),
                              "estacao": estacao, "valor_original": bruto.at[i, estacao]})
        numericos[estacao] = valores.mask(invalido)
        ausentes[estacao] = int(falta.sum())
    if invalidos:
        mensagem = f"{len(invalidos)} valor(es) inválido(s) detectado(s) no CSV."
        if val["acao_valores_invalidos"] == "interromper":
            falhar(estado, "valores_invalidos", mensagem + " Execução interrompida; veja auditoria.", invalidos)
        registrar(estado, "warning", "valores_marcados_nan", mensagem + " Marcados NaN apenas na cópia de trabalho, por configuração explícita.", invalidos)
    registrar(estado, "info", "ausencias_originais", "Marcadores de ausência contabilizados por estação (arquivo inteiro).", ausentes)
    numericos.index = pd.DatetimeIndex(datas, name="data")
    if not numericos.index.is_monotonic_increasing:
        registrar(estado, "info", "ordenacao", "Registros ordenados cronologicamente na cópia de trabalho.")
    # O hash permite identificar exatamente a entrada usada, sem modificá-la.
    with caminho.open("rb") as arquivo:
        estado["dados"]["sha256_csv"] = hashlib.file_digest(arquivo, "sha256").hexdigest()
    registrar(estado, "info", "leitura_concluida", f"CSV carregado: {len(numericos)} dias e {len(estacoes)} estações.")
    return numericos.sort_index()


dados_brutos_validados = carregar_dados(config, auditoria)
try:
    estacoes_avaliadas = resolver_estacoes_avaliacao(config["evaluation_stations"], dados_brutos_validados.columns)
except ValueError as exc:
    falhar(auditoria, "estacoes_alvo_invalidas", str(exc))
config["_runtime"]["estacoes_avaliadas"] = estacoes_avaliadas
registrar(auditoria, "info", "estacoes_alvo", f"{len(estacoes_avaliadas)} estação(ões)-alvo selecionada(s).",
          {"alvos": estacoes_avaliadas, "universo_de_estacoes": list(dados_brutos_validados.columns)})
display(dados_brutos_validados.head())


INFO | 16437 datas válidas e sem duplicatas no CSV.
INFO | Marcadores de ausência contabilizados por estação (arquivo inteiro).
INFO | CSV carregado: 16437 dias e 19 estações.
INFO | 19 estação(ões)-alvo selecionada(s).


,2751007,2751017,2851003,2851005,2851020,2851021,2851022,2851024,2851028,2851043,2851044,2852004,2852014,2852016,2852031,2852046,2951010,2951070,83914
data,,,,,,,,,,,,,,,,,,,
1980-01-01,0.0,0.0,4.3,0.0,0.0,0.0,0.0,0.0,18.6,0.8,NaN,0.0,1.2,0.0,0.0,0.2,0.0,NaN,0.7
1980-01-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0
1980-01-03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0
1980-01-04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0
1980-01-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0


## 4. Controle da estrutura temporal

O intervalo é inclusivo. Datas inteiramente ausentes são listadas na auditoria,
inclusive no começo/fim solicitado. Se o período não intersectar nenhuma data do
arquivo, a execução para. Se houver cobertura parcial ou intervalos vazios dentro
da série, a análise prossegue com avisos e lacunas explícitas.


In [18]:
def controlar_calendario(dados, inicio, fim, estado):
    """Filtra período e explicita dias ausentes com NaN; retorna calendário diário completo."""
    if fim < dados.index.min() or inicio > dados.index.max():
        falhar(estado, "periodo_sem_intersecao", "O período solicitado está totalmente fora das datas do CSV.",
               {"inicio_csv": str(dados.index.min()), "fim_csv": str(dados.index.max())})
    recorte = dados.loc[inicio:fim].copy()
    calendario = pd.date_range(inicio, fim, freq="D", name="data")
    ausentes = calendario.difference(recorte.index)
    if inicio < dados.index.min() or fim > dados.index.max():
        registrar(estado, "warning", "cobertura_parcial", "Parte do período solicitado está fora da cobertura temporal do CSV.")
    if ausentes.size:
        registrar(estado, "warning", "datas_ausentes", f"{len(ausentes)} dia(s) sem registro no período; inseridos como NaN, sem imputação.",
                  [str(d.date()) for d in ausentes])
    else:
        registrar(estado, "info", "calendario_completo", "Nenhuma data ausente no calendário diário solicitado.")
    if inicio.day != 1 or fim.day != fim.days_in_month:
        registrar(estado, "warning", "meses_limite_parciais", "O período corta um mês civil. Totais de meses parciais permanecerão NaN.")
    return recorte.reindex(calendario)


diario = controlar_calendario(dados_brutos_validados, config['_runtime']['inicio'], config['_runtime']['fim'], auditoria)
print(f"Calendário analisado: {len(diario)} dias; {diario.index.min().date()} a {diario.index.max().date()}.")


INFO | Nenhuma data ausente no calendário diário solicitado.


Calendário analisado: 16437 dias; 1980-01-01 a 2024-12-31.


## 5. Conversão diária → mensal

Para estação *s* e mês *m*, a soma só é aceita quando a contagem de dias válidos é
igual ao número de dias do mês civil. `days_in_month` trata automaticamente os
anos bissextos. `min_count=1` também impede transformar uma série toda ausente em
zero, antes mesmo da aplicação da máscara de completude.


In [19]:
def agregar_precipitacao_mensal(diario):
    """Agrega calendário diário único; retorna totais mensais completos, contagens e dias esperados."""
    if diario.index.has_duplicates:
        raise ValueError("Agregação requer datas únicas; duplicatas não são somadas.")
    soma = diario.resample("MS").sum(min_count=1)
    contagem = diario.resample("MS").count()
    esperados = pd.Series(soma.index.days_in_month, index=soma.index, name="dias_esperados")
    mensal = soma.where(contagem.eq(esperados, axis=0))
    mensal.index.name = "data"
    return mensal, contagem, esperados


mensal, dias_validos_mensais, dias_esperados_mensais = agregar_precipitacao_mensal(diario)
display(mensal.head())


,2751007,2751017,2851003,2851005,2851020,2851021,2851022,2851024,2851028,2851043,2851044,2852004,2852014,2852016,2852031,2852046,2951010,2951070,83914
data,,,,,,,,,,,,,,,,,,,
1980-01-01,202.7,135.6,140.1,128.0,137.2,52.1,128.0,119.2,135.9,114.0,NaN,114.4,145.6,157.4,126.2,146.4,102.4,NaN,118.9
1980-02-01,55.7,100.8,176.8,131.6,71.7,124.5,119.8,182.2,46.6,73.2,NaN,101.2,141.1,102.8,59.0,106.3,160.5,NaN,135.2
1980-03-01,142.6,60.9,72.1,127.4,93.7,29.1,171.0,83.6,126.6,89.8,NaN,86.9,67.7,82.0,99.4,110.2,48.2,NaN,167.9
1980-04-01,56.0,87.0,51.0,68.8,44.6,58.1,122.6,86.4,44.0,95.2,NaN,66.6,75.1,61.2,57.6,47.5,61.1,NaN,29.0
1980-05-01,186.1,202.1,156.8,207.8,174.5,135.0,173.2,174.3,177.3,173.8,NaN,171.0,155.4,204.8,237.0,162.2,159.0,NaN,192.2


## 6. Análise de disponibilidade

`inicio_disponivel` e `fim_disponivel` são o primeiro e o último **mês completo**.
As colunas `inicio_diario_disponivel` e `fim_diario_disponivel` descrevem os dados
diários válidos dentro do período. Estações sem meses completos permanecem nas
tabelas e matrizes; não são descartadas.


In [20]:
def calcular_disponibilidade(mensal, diario, alerta, estado):
    """Resume completude por estação; recebe séries/limiar/auditoria e retorna tabela."""
    linhas = []
    for estacao in mensal.columns:
        validos = mensal[estacao].dropna()
        dias = diario[estacao].dropna()
        total, n = len(mensal), len(validos)
        linhas.append({"estacao": estacao,
            "inicio_disponivel": validos.index.min().strftime("%Y-%m") if n else None,
            "fim_disponivel": validos.index.max().strftime("%Y-%m") if n else None,
            "inicio_diario_disponivel": dias.index.min().strftime("%Y-%m-%d") if len(dias) else None,
            "fim_diario_disponivel": dias.index.max().strftime("%Y-%m-%d") if len(dias) else None,
            "meses_esperados": total, "meses_completos": n, "meses_com_falha": total - n,
            "percentual_meses_completos": 100 * n / total,
            "percentual_meses_ausentes": 100 * (total - n) / total})
        if not len(dias):
            registrar(estado, "warning", "estacao_sem_dados", f"Estação {estacao}: nenhum dado diário válido no período.")
        elif not n:
            registrar(estado, "warning", "estacao_sem_meses", f"Estação {estacao}: há dados diários, mas nenhum mês completo.")
        elif n < alerta:
            registrar(estado, "warning", "poucos_meses", f"Estação {estacao}: somente {n} meses completos; alerta configurado = {alerta}.")
    return pd.DataFrame(linhas)


disponibilidade = calcular_disponibilidade(mensal, diario, config['validacao']['meses_completos_alerta'], auditoria)
display(disponibilidade)


,estacao,inicio_disponivel,fim_disponivel,inicio_diario_disponivel,fim_diario_disponivel,meses_esperados,meses_completos,meses_com_falha,percentual_meses_completos,percentual_meses_ausentes
0,2751007,1980-01,2024-12,1980-01-01,2024-12-31,540,529,11,97.962963,2.037037
1,2751017,1980-01,2024-11,1980-01-01,2024-12-31,540,511,29,94.629630,5.370370
2,2851003,1980-01,2024-12,1980-01-01,2024-12-31,540,529,11,97.962963,2.037037
3,2851005,1980-01,2012-12,1980-01-01,2012-12-31,540,396,144,73.333333,26.666667
4,2851020,1980-01,2024-12,1980-01-01,2024-12-31,540,502,38,92.962963,7.037037
5,2851021,1980-01,2024-12,1980-01-01,2024-12-31,540,524,16,97.037037,2.962963
6,2851022,1980-01,2024-12,1980-01-01,2024-12-31,540,518,22,95.925926,4.074074
7,2851024,1980-01,2024-12,1980-01-01,2024-12-31,540,517,23,95.740741,4.259259
8,2851028,1980-01,2024-12,1980-01-01,2024-12-31,540,498,42,92.222222,7.777778
9,2851043,1980-01,2024-12,1980-01-01,2024-12-31,540,526,14,97.407407,2.592593


## 7. Visualização das séries mensais

Clique na legenda para ocultar/exibir uma estação; use zoom ou pan na barra de
ferramentas. `connectgaps=False` preserva as lacunas. Marcadores tornam meses
observados isolados visíveis. O HTML final incorpora o Plotly e funciona offline.


In [21]:
def gerar_grafico_series(mensal, cfg):
    """Cria gráfico Plotly de séries com lacunas; recebe mensal/configuração e retorna Figure."""
    fig = go.Figure()
    casas = cfg["visualizacao"]["casas_decimais"]
    for estacao in mensal.columns:
        fig.add_trace(go.Scatter(x=mensal.index, y=mensal[estacao], name=escape(str(estacao)),
            mode="lines+markers", marker={"size": 3}, connectgaps=False,
            hovertemplate="Estação: %{fullData.name}<br>Mês: %{x|%Y-%m}<br>Precipitação: %{y:." + str(casas) + "f} mm<extra></extra>"))
    fig.update_layout(title=escape(cfg["projeto"]["nome"]) + " — totais mensais completos",
        template=cfg["visualizacao"]["template"], height=cfg["visualizacao"]["altura_series"],
        xaxis_title="Mês", yaxis_title="Precipitação mensal (mm)", hovermode="closest",
        legend_title="Estação", xaxis={"rangeslider": {"visible": True}}, dragmode="zoom")
    return fig


config_plotly = {"scrollZoom": config["visualizacao"]["scroll_zoom"], "displaylogo": False, "responsive": True}
fig_series = gerar_grafico_series(mensal, config)
if config["visualizacao"]["exibir_no_notebook"]:
    fig_series.show(config=config_plotly)


## 8. Matriz de correlação

O pandas calcula Pearson usando observações completas **por par**. Não se aplica
`dropna()` ao conjunto de todas as estações. Séries constantes são verificadas
também na interseção do par, pois uma estação variável no conjunto inteiro pode
ser constante nos meses em comum com outra estação.

O diagnóstico exportado distingue pares calculáveis, pares sem observações
suficientes e pares com série constante. Para pares calculáveis abaixo do mínimo
de seleção, mantém-se `r` para consulta, com indicação de inelegibilidade.


In [22]:
def calcular_matrizes_pareadas(mensal, minimo_apoio, estado):
    """Calcula Pearson, contagens e diagnóstico pareados; retorna (r, n, diagnóstico)."""
    mascara = mensal.notna().astype("int64")
    n = mascara.T.dot(mascara).astype("int64")
    r = mensal.corr(method="pearson", min_periods=2)
    linhas = []
    estacoes = list(mensal.columns)
    for i, a in enumerate(estacoes):
        for b in estacoes[i + 1:]:
            pares = mensal[[a, b]].dropna()
            quantidade = int(n.loc[a, b])
            motivo = "calculavel"
            if quantidade < 2:
                motivo = "menos_de_dois_meses_coincidentes"
            elif pares[a].nunique() < 2 or pares[b].nunique() < 2:
                motivo = "serie_constante_na_intersecao"
            elif not np.isfinite(r.loc[a, b]):
                motivo = "resultado_nao_finito"
            if motivo != "calculavel":
                r.loc[a, b] = r.loc[b, a] = np.nan
            linhas.append({"estacao_a": a, "estacao_b": b, "correlacao_r": r.loc[a, b],
                "r2": r.loc[a, b] ** 2, "meses_coincidentes": quantidade,
                "atinge_minimo_meses_apoio": quantidade >= minimo_apoio, "diagnostico": motivo})
    diagnostico = pd.DataFrame(linhas, columns=["estacao_a", "estacao_b", "correlacao_r", "r2",
        "meses_coincidentes", "atinge_minimo_meses_apoio", "diagnostico"])
    for i in range(len(r)):
        r.iat[i, i] = 1.0  # Convenção de apresentação, nunca critério de seleção.
    r.index.name = n.index.name = "estacao"
    ruins = diagnostico.loc[diagnostico["diagnostico"] != "calculavel"]
    if len(ruins):
        detalhes = ruins.astype(object).where(ruins.notna(), None).to_dict("records")
        registrar(estado, "warning", "pares_indefinidos", f"{len(ruins)} par(es) sem correlação calculável; consulte diagnostico_pares.csv.", detalhes)
    registrar(estado, "info", "correlacoes", f"{len(linhas)} par(es) distintos analisados; diagonal fixada em 1 por convenção.")
    return r, n, diagnostico


matriz_r, matriz_n, diagnostico_pares = calcular_matrizes_pareadas(
    mensal, config["selecao_apoio"]["meses_coincidentes_minimos"], auditoria)
display(matriz_r)


INFO | 171 par(es) distintos analisados; diagonal fixada em 1 por convenção.


,2751007,2751017,2851003,2851005,2851020,2851021,2851022,2851024,2851028,2851043,2851044,2852004,2852014,2852016,2852031,2852046,2951010,2951070,83914
estacao,,,,,,,,,,,,,,,,,,,
2751007,1.000000,0.900560,0.743714,0.766574,0.842247,0.684808,0.775655,0.738034,0.810655,0.878377,0.711075,0.718871,0.698991,0.784832,0.788984,0.898935,0.532237,0.650539,0.818495
2751017,0.900560,1.000000,0.730287,0.754792,0.812377,0.707752,0.749454,0.733870,0.775302,0.860219,0.706812,0.692749,0.684918,0.773071,0.758032,0.862753,0.515635,0.613219,0.786352
2851003,0.743714,0.730287,1.000000,0.791617,0.785352,0.895908,0.862467,0.922021,0.777727,0.731040,0.892557,0.828940,0.832232,0.797385,0.796411,0.742030,0.704812,0.858154,0.758808
2851005,0.766574,0.754792,0.791617,1.000000,0.852656,0.761307,0.884582,0.831539,0.841750,0.721173,0.805197,0.790702,0.785279,0.864732,0.837313,0.776376,0.627518,0.704041,0.791383
2851020,0.842247,0.812377,0.785352,0.852656,1.000000,0.750969,0.851653,0.805323,0.885297,0.795312,0.762062,0.782312,0.740948,0.858021,0.857648,0.863215,0.572816,0.691412,0.874124
2851021,0.684808,0.707752,0.895908,0.761307,0.750969,1.000000,0.825708,0.904453,0.751510,0.683007,0.898144,0.809208,0.827222,0.764511,0.777118,0.697465,0.701069,0.841267,0.722751
2851022,0.775655,0.749454,0.862467,0.884582,0.851653,0.825708,1.000000,0.894523,0.827017,0.737394,0.855639,0.834140,0.837641,0.869478,0.847257,0.782650,0.680835,0.797445,0.804669
2851024,0.738034,0.733870,0.922021,0.831539,0.805323,0.904453,0.894523,1.000000,0.821384,0.716083,0.923887,0.861680,0.871874,0.841677,0.835531,0.755323,0.725140,0.860376,0.785098
2851028,0.810655,0.775302,0.777727,0.841750,0.885297,0.751510,0.827017,0.821384,1.000000,0.767740,0.795481,0.790234,0.763021,0.844966,0.828818,0.815537,0.572663,0.712357,0.823891


## 9. Matriz de R²

A operação é somente o quadrado de Pearson. `NaN` continua `NaN`. Uma correlação
negativa pode ter R² alto; o ranking usa o **r com seu sinal**.


In [23]:
matriz_r2 = matriz_r.pow(2)
display(matriz_r2)


,2751007,2751017,2851003,2851005,2851020,2851021,2851022,2851024,2851028,2851043,2851044,2852004,2852014,2852016,2852031,2852046,2951010,2951070,83914
estacao,,,,,,,,,,,,,,,,,,,
2751007,1.000000,0.811008,0.553111,0.587636,0.709379,0.468962,0.601641,0.544694,0.657162,0.771546,0.505628,0.516775,0.488588,0.615962,0.622495,0.808084,0.283276,0.423201,0.669934
2751017,0.811008,1.000000,0.533319,0.569711,0.659956,0.500913,0.561682,0.538565,0.601093,0.739977,0.499583,0.479902,0.469113,0.597638,0.574612,0.744343,0.265879,0.376037,0.618350
2851003,0.553111,0.533319,1.000000,0.626657,0.616778,0.802652,0.743849,0.850122,0.604860,0.534419,0.796657,0.687142,0.692610,0.635823,0.634270,0.550608,0.496760,0.736428,0.575789
2851005,0.587636,0.569711,0.626657,1.000000,0.727023,0.579588,0.782485,0.691456,0.708544,0.520090,0.648342,0.625209,0.616663,0.747761,0.701092,0.602759,0.393778,0.495674,0.626288
2851020,0.709379,0.659956,0.616778,0.727023,1.000000,0.563954,0.725313,0.648546,0.783751,0.632522,0.580739,0.612012,0.549004,0.736200,0.735561,0.745140,0.328118,0.478050,0.764093
2851021,0.468962,0.500913,0.802652,0.579588,0.563954,1.000000,0.681793,0.818035,0.564768,0.466498,0.806663,0.654817,0.684297,0.584477,0.603913,0.486458,0.491498,0.707730,0.522369
2851022,0.601641,0.561682,0.743849,0.782485,0.725313,0.681793,1.000000,0.800171,0.683957,0.543749,0.732119,0.695790,0.701643,0.755992,0.717844,0.612540,0.463537,0.635918,0.647492
2851024,0.544694,0.538565,0.850122,0.691456,0.648546,0.818035,0.800171,1.000000,0.674671,0.512775,0.853567,0.742492,0.760165,0.708421,0.698112,0.570513,0.525829,0.740247,0.616379
2851028,0.657162,0.601093,0.604860,0.708544,0.783751,0.564768,0.683957,0.674671,1.000000,0.589425,0.632790,0.624469,0.582201,0.713967,0.686939,0.665100,0.327943,0.507452,0.678796


## 10. Matriz de meses coincidentes

A contagem usa a multiplicação da máscara de validade em inteiros de 64 bits:
`M.T @ M`. Cada célula conta meses válidos simultaneamente; a diagonal é o número
de meses completos da própria estação. O uso de inteiros evita contagem booleana
ou estouro em séries com centenas de meses.


In [24]:
display(matriz_n)


,2751007,2751017,2851003,2851005,2851020,2851021,2851022,2851024,2851028,2851043,2851044,2852004,2852014,2852016,2852031,2852046,2951010,2951070,83914
estacao,,,,,,,,,,,,,,,,,,,
2751007,529,507,523,385,491,518,512,511,487,521,412,506,489,385,482,488,462,416,449
2751017,507,511,505,380,473,500,495,493,470,502,394,488,472,380,466,474,447,397,434
2851003,523,505,529,388,491,518,512,512,487,520,409,506,489,388,482,489,464,414,449
2851005,385,380,388,396,396,382,384,378,396,387,280,396,396,396,396,389,368,278,328
2851020,491,473,491,396,502,488,483,483,489,489,383,501,491,396,487,459,436,384,422
2851021,518,500,518,382,488,524,508,508,484,515,405,502,486,382,479,481,460,410,445
2851022,512,495,512,384,483,508,518,500,480,509,401,498,482,384,475,480,453,405,439
2851024,511,493,512,378,483,508,500,517,479,508,403,496,481,378,475,476,450,403,439
2851028,487,470,487,396,489,484,480,479,498,487,377,497,496,396,490,453,432,380,419


## 11. Seleção e ranking das estações de apoio

Aceita-se uma candidata quando `r >= correlacao_minima` **e**
`n >= meses_coincidentes_minimos`. A estação-alvo nunca apoia a si própria.
Ordenação: `r` decrescente; empate exato em `r` é resolvido por `n` decrescente e
depois por nome de estação crescente. Os limiares usam valores sem arredondamento.
Após os filtros e a ordenação, `n_support_stations` limita os apoios por alvo;
`null` mantém todos. Uma estação fora de `evaluation_stations` continua candidata
a apoio se estiver no universo carregado. O CSV mantém suas seis colunas originais:
`estacao_alvo`, `ranking`, `estacao_apoio`, `correlacao_r`, `r2`, `meses_coincidentes`.
Se houver menos de N elegíveis, são retornadas somente as elegíveis, com aviso.
Se nenhuma candidata passar, o CSV conserva seu cabeçalho e o resumo informa isso.


In [25]:
def selecionar_estacoes_apoio(r, r2, n, criterios, evaluation_stations="all", n_support_stations=None):
    """Filtra/ordena por alvo; recebe matrizes, critérios, alvos e limite; retorna ranking detalhado."""
    colunas = ["estacao_alvo", "ranking", "estacao_apoio", "correlacao_r", "r2", "meses_coincidentes"]
    linhas = []
    if n_support_stations is not None and (type(n_support_stations) is not int or n_support_stations < 1):
        raise ValueError("n_support_stations deve ser null ou um inteiro >= 1.")
    for alvo in resolver_estacoes_avaliacao(evaluation_stations, r.index):
        candidatas = []
        for apoio in r.columns:
            valor, quantidade = r.loc[alvo, apoio], int(n.loc[alvo, apoio])
            if (apoio != alvo and np.isfinite(valor) and valor >= criterios["correlacao_minima"]
                    and quantidade >= criterios["meses_coincidentes_minimos"]):
                candidatas.append({"estacao_alvo": alvo, "estacao_apoio": apoio,
                    "correlacao_r": float(valor), "r2": float(r2.loc[alvo, apoio]), "meses_coincidentes": quantidade})
        candidatas.sort(key=lambda linha: (-linha["correlacao_r"], -linha["meses_coincidentes"], linha["estacao_apoio"]))
        if n_support_stations is not None:
            candidatas = candidatas[:n_support_stations]
        for ranking, linha in enumerate(candidatas, start=1):
            linha["ranking"] = ranking
            linhas.append(linha)
    return pd.DataFrame(linhas, columns=colunas)


apoios = selecionar_estacoes_apoio(matriz_r, matriz_r2, matriz_n, config["selecao_apoio"],
    evaluation_stations=estacoes_avaliadas, n_support_stations=config["n_support_stations"])
sem_apoio = [s for s in estacoes_avaliadas if s not in set(apoios["estacao_alvo"])]
if sem_apoio:
    registrar(auditoria, "warning", "estacoes_sem_apoio", f"{len(sem_apoio)} estação(ões) sem apoio elegível: {', '.join(sem_apoio)}.", sem_apoio)
if config["n_support_stations"] is not None:
    for alvo in estacoes_avaliadas:
        quantidade = int(apoios["estacao_alvo"].eq(alvo).sum())
        if 0 < quantidade < config["n_support_stations"]:
            registrar(auditoria, "warning", "apoios_insuficientes",
                f"Estação {alvo}: {quantidade} de {config['n_support_stations']} apoios solicitados são elegíveis; critérios preservados.")
display(apoios)


,estacao_alvo,ranking,estacao_apoio,correlacao_r,r2,meses_coincidentes
0,2751007,1,2751017,0.900560,0.811008,507
1,2751007,2,2852046,0.898935,0.808084,488
2,2751007,3,2851043,0.878377,0.771546,521
3,2751007,4,2851020,0.842247,0.709379,491
4,2751007,5,83914,0.818495,0.669934,449
...,...,...,...,...,...,...
141,83914,3,2852031,0.866834,0.751402,415
142,83914,4,2852046,0.861916,0.742899,421
143,83914,5,2851028,0.823891,0.678796,419
144,83914,6,2751007,0.818495,0.669934,449


## 12. Heatmaps

Os três gráficos apresentam **r + R² + n** juntos no hover. A anotação numérica
das células é ativada até o número de estações definido no YAML; acima dele,
os valores continuam disponíveis no hover. Lacunas indicam correlação indefinida,
não correlação zero. Para inspecionar um par indefinido, use o heatmap de contagens
ou o diagnóstico CSV. Os eixos usam categorias textuais para preservar códigos.


In [26]:
def gerar_heatmap(matriz, tipo, r, r2, n, diagnostico, cfg):
    """Cria heatmap com r/R²/n e diagnóstico no hover; recebe matrizes/config e retorna Figure."""
    vis = cfg["visualizacao"]
    especificacoes = {
        "r": ("Correlação de Pearson (r)", vis["escala_correlacao"], -1, 1),
        "r2": ("Coeficiente de determinação (R² = r²)", vis["escala_r2"], 0, 1),
        "n": ("Meses coincidentes (n)", vis["escala_meses"], 0, max(1, int(n.to_numpy().max()))),
    }
    titulo, escala, zmin, zmax = especificacoes[tipo]
    status = {}
    for linha in diagnostico.itertuples(index=False):
        valor = linha.diagnostico
        if valor == "calculavel" and not linha.atinge_minimo_meses_apoio:
            valor = "calculável, mas abaixo do mínimo de meses para apoio"
        status[linha.estacao_a, linha.estacao_b] = status[linha.estacao_b, linha.estacao_a] = valor
    extras = np.empty((len(matriz), len(matriz), 4), dtype=object)
    casas = vis["casas_decimais"]
    for i, a in enumerate(matriz.index):
        for j, b in enumerate(matriz.columns):
            extras[i, j] = [f"{r.loc[a, b]:.{casas}f}" if pd.notna(r.loc[a, b]) else "indefinido",
                f"{r2.loc[a, b]:.{casas}f}" if pd.notna(r2.loc[a, b]) else "indefinido",
                str(n.loc[a, b]), "diagonal: r = R² = 1 por convenção" if a == b else status[a, b]]
    formato = ".0f" if tipo == "n" else f".{casas}f"
    anotacao = "%{z:" + formato + "}" if len(matriz) <= vis["mostrar_valores_ate_n_estacoes"] else None
    fig = go.Figure(go.Heatmap(z=matriz.to_numpy(), x=[escape(str(s)) for s in matriz.columns],
        y=[escape(str(s)) for s in matriz.index], zmin=zmin, zmax=zmax, colorscale=escala,
        customdata=extras, texttemplate=anotacao, hoverongaps=False,
        hovertemplate="Estação A: %{y}<br>Estação B: %{x}<br>r: %{customdata[0]}<br>R²: %{customdata[1]}<br>Meses coincidentes: %{customdata[2]}<br>%{customdata[3]}<extra></extra>"))
    fig.update_layout(title=titulo, template=vis["template"], height=vis["altura_heatmap"],
                      xaxis_title="Estação", yaxis_title="Estação")
    fig.update_xaxes(type="category", side="bottom", automargin=True)
    fig.update_yaxes(type="category", autorange="reversed", automargin=True)
    return fig


fig_r = gerar_heatmap(matriz_r, "r", matriz_r, matriz_r2, matriz_n, diagnostico_pares, config)
fig_r2 = gerar_heatmap(matriz_r2, "r2", matriz_r, matriz_r2, matriz_n, diagnostico_pares, config)
fig_n = gerar_heatmap(matriz_n, "n", matriz_r, matriz_r2, matriz_n, diagnostico_pares, config)
if config["visualizacao"]["exibir_no_notebook"]:
    for figura in [fig_r, fig_r2, fig_n]:
        figura.show(config=config_plotly)


## 13. Exportação dos resultados

Os CSVs conservam a precisão numérica, sem arredondar para tomar decisões.
O índice mensal é exportado como `AAAA-MM`. Ausências são escritas conforme o
YAML (`NaN` por padrão). HTMLs completos incorporam JavaScript e dispensam internet;
por isso, cada HTML pode ocupar alguns megabytes.

Todas as saídas vão para a mesma pasta exclusiva de execução. A cópia do YAML
registra a pasta-base, para que uma futura execução crie uma pasta irmã; a auditoria
registra o caminho exato desta execução.

Além das dez saídas solicitadas, são gravados `diagnostico_pares.csv`,
`auditoria_execucao.json`, `execucao.log` e `config_utilizada.yaml` para rastreabilidade.


In [27]:
def salvar_resultados(tabelas, figuras, cfg, estado):
    """Exporta tabelas, HTMLs e YAML usado; recebe produtos/config/auditoria e retorna caminhos."""
    destinos = cfg["_runtime"]["destinos"]
    opcoes = cfg["saida"]
    try:
        for chave, (tabela, usar_indice) in tabelas.items():
            exportar = tabela.copy()
            if chave == "mensal":
                exportar.index = exportar.index.strftime("%Y-%m")
                exportar.index.name = "data"
            exportar.to_csv(destinos[chave], index=usar_indice, sep=opcoes["separador_csv"],
                decimal=opcoes["decimal_csv"], encoding=opcoes["encoding_csv"], na_rep=opcoes["representacao_nan"])
        for chave, figura in figuras.items():
            figura.write_html(str(destinos[chave]), include_plotlyjs=True, full_html=True, auto_open=False,
                config={"scrollZoom": cfg["visualizacao"]["scroll_zoom"], "displaylogo": False, "responsive": True})
        copia_cfg = deepcopy({k: v for k, v in cfg.items() if k != "_runtime"})
        copia_cfg["entrada"]["arquivo_csv"] = str(cfg["_runtime"]["entrada"])
        copia_cfg["saida"]["diretorio"] = str(cfg["_runtime"]["saida_base"])  # Próxima execução será uma pasta irmã.
        destinos["configuracao"].write_text(yaml.safe_dump(copia_cfg,
            sort_keys=False, allow_unicode=True), encoding="utf-8")
    except Exception as exc:
        registrar(estado, "error", "erro_exportacao", f"Exportação incompleta: {exc}")
        raise
    estado["dados"]["status"] = "concluido"
    estado["dados"]["fim_utc"] = datetime.now(timezone.utc).isoformat()
    estado["dados"]["arquivos"] = {k: str(v) for k, v in destinos.items()}
    registrar(estado, "info", "conclusao", "Etapa 1 concluída; tabelas, gráficos e auditoria exportados.")
    return destinos


caminhos_resultados = salvar_resultados(
    tabelas={"mensal": (mensal, True), "disponibilidade": (disponibilidade, False),
        "correlacao": (matriz_r, True), "r2": (matriz_r2, True), "meses_coincidentes": (matriz_n, True),
        "estacoes_apoio": (apoios, False), "diagnostico_pares": (diagnostico_pares, False)},
    figuras={"series_html": fig_series, "correlacao_html": fig_r, "r2_html": fig_r2, "meses_html": fig_n},
    cfg=config, estado=auditoria)


INFO | Etapa 1 concluída; tabelas, gráficos e auditoria exportados.


## 14. Resumo final da execução

Os valores apresentados são arredondados somente para leitura; os CSVs e a
seleção utilizam a precisão original. O resumo apresenta a contagem e a tabela
completa de apoios de cada alvo, sem truncar as linhas. Os apoios são candidatos para trabalho
posterior, não séries já preenchidas ou aprovadas por dupla massa.


In [28]:
def imprimir_resumo(mensal, apoios, cfg):
    """Imprime período, critérios e ranking por alvo; recebe resultados/config e retorna None."""
    criterios, rt = cfg["selecao_apoio"], cfg["_runtime"]
    casas = cfg["visualizacao"]["casas_decimais"]
    print("=" * 60)
    print("ETAPA 1 — CORRELAÇÃO E DISPONIBILIDADE")
    print("=" * 60)
    print(f"Período solicitado: {rt['inicio']:%Y-%m-%d} a {rt['fim']:%Y-%m-%d}")
    alvos = resolver_estacoes_avaliacao(cfg["evaluation_stations"], mensal.columns)
    print(f"Bacia: {rt['id_bacia']} | Execução: {rt['instante_execucao']}")
    print(f"Estações disponíveis: {mensal.shape[1]} | Estações-alvo avaliadas: {len(alvos)}")
    print(f"Número total de meses civis: {len(mensal)}")
    print(f"Critérios: r >= {criterios['correlacao_minima']}; meses coincidentes >= {criterios['meses_coincidentes_minimos']}")
    for estacao in alvos:
        selecionadas = apoios.loc[apoios["estacao_alvo"] == estacao]
        limite = cfg["n_support_stations"]
        solicitado = "todos os elegíveis" if limite is None else f"até {limite}"
        print(f"\nEstação-alvo {estacao}: {len(selecionadas)} apoio(s) selecionado(s); solicitado: {solicitado}.")
        if selecionadas.empty:
            print("Nenhuma estação de apoio atende simultaneamente aos critérios.")
        else:
            print(selecionadas.to_string(index=False, formatters={
                "correlacao_r": lambda x: f"{x:.{casas}f}", "r2": lambda x: f"{x:.{casas}f}"}))
    print(f"\nResultados: {rt['saida']}")
    print("=" * 60)


imprimir_resumo(mensal, apoios, config)


ETAPA 1 — CORRELAÇÃO E DISPONIBILIDADE
Período solicitado: 1980-01-01 a 2024-12-31
Bacia: 86500000 | Execução: 2026-09-28T13:10:52.943295-03:00
Estações disponíveis: 19 | Estações-alvo avaliadas: 19
Número total de meses civis: 540
Critérios: r >= 0.8; meses coincidentes >= 120

Estação-alvo 2751007: 6 apoio(s) selecionado(s); solicitado: todos os elegíveis.
estacao_alvo  ranking estacao_apoio correlacao_r    r2  meses_coincidentes
     2751007        1       2751017        0.901 0.811                 507
     2751007        2       2852046        0.899 0.808                 488
     2751007        3       2851043        0.878 0.772                 521
     2751007        4       2851020        0.842 0.709                 491
     2751007        5         83914        0.818 0.670                 449
     2751007        6       2851028        0.811 0.657                 487

Estação-alvo 2751017: 4 apoio(s) selecionado(s); solicitado: todos os elegíveis.
estacao_alvo  ranking estacao_ap

### Referências de implementação

- [pandas — Pearson e observações completas por par](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html).
- [pandas — resample](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.resample.html).
- [pandas — número de dias do mês](https://pandas.pydata.org/docs/reference/api/pandas.DatetimeIndex.days_in_month.html).
- [Plotly — exportação HTML independente](https://plotly.com/python/interactive-html-export/).

Entrega verificada com dados sintéticos; veja `VALIDACAO.md`. O CSV real não foi
fornecido nesta solicitação. Não há resultados empíricos da bacia nesta entrega.
